In [6]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer
from langdetect import detect

plt.rcParams["figure.figsize"] = (20, 13)
%matplotlib inline
%config InlineBackend.figure_format = "retina"

In [37]:
interactions = pd.read_csv("../data_final_project/KuaiRec/data/big_matrix.csv")
small_interactions = pd.read_csv("../data_final_project/KuaiRec/data/small_matrix.csv")
captions = pd.read_csv("../data_final_project/KuaiRec/data/kuairec_caption_category.csv", lineterminator='\n')

def clean_df(df):
    df = df.dropna()
    df = df.drop_duplicates()  
    return df  

def clean_df_timestamp(df):
    df = clean_df(df)
    df = df[df["timestamp"] >= 0]
    return df

captions = clean_df(captions)
captions = captions.drop_duplicates(subset='video_id')
train_df = clean_df_timestamp(interactions)
test_df = clean_df_timestamp(small_interactions)

In [38]:
captions.head()

,video_id,manual_cover_text,caption,topic_tag,first_level_category_id,first_level_category_name,second_level_category_id,second_level_category_name,third_level_category_id,third_level_category_name
0,0,UNKNOWN,精神小伙路难走 程哥你狗粮慢点撒,[],8,颜值,673,颜值随拍,-124,UNKNOWN
2,2,UNKNOWN,晚饭后，运动一下！,[],9,喜剧,727,搞笑互动,-124,UNKNOWN
3,3,UNKNOWN,我平淡无奇，惊艳不了时光，温柔不了岁月，我只想漫无目的的走走，努力发笔小财，给自己买花 自己长大.,[],26,摄影,686,主题摄影,2434,景物摄影
4,4,五爱街最美美女 一天1q,#搞笑 #感谢快手我要上热门 #五爱市场 这真是完美搭配啊！,"[五爱市场,感谢快手我要上热门,搞笑]",5,时尚,737,营销售卖,2596,女装
5,5,UNKNOWN,“你们吵的越狠 他们的手就握的越紧” #文轩 #刘耀文 #宋亚轩 #顾子璇...,"[刘耀文,宋亚轩,文轩,顾子璇是樱桃吖,顾子璇超级喜欢文轩]",6,明星娱乐,667,娱乐八卦,2375,饭制


In [39]:
captions = captions[['video_id', 'caption', 'first_level_category_name', 'second_level_category_name','third_level_category_name']]
videos_id = interactions['video_id']
captions = pd.merge(videos_id, captions, on='video_id', how='left')

In [40]:
captions.drop_duplicates(subset='video_id', inplace=True)
captions[captions['video_id'] == 1]
captions.reset_index(inplace=True)
captions.fillna('UNKNOWN', inplace=True)

In [41]:
def detect_language(text):
    try:
        return detect(text)
    except:
        return "UNKNOWN"
    
captions['language'] = captions['caption'].apply(detect_language)

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer

# Load the BERT tokenizers
tokenizer_cn = BertTokenizer.from_pretrained("bert-base-chinese")
tokenizer_kr = BertTokenizer.from_pretrained("beomi/kcbert-base")

# Tokenize the text based on language
def tokenize_by_lang(text, lang):
    text = str(text)
    if lang == 'zh-cn':
        return ' '.join(tokenizer_cn.tokenize(text))
    elif lang == 'ko':
        return ' '.join(tokenizer_kr.tokenize(text))
    else:
        return 'UNKNOWN'

captions['tokenized'] = captions.apply(lambda row: tokenize_by_lang(row['caption'], row['language']), axis=1)

tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(captions['tokenized'])

text_sim = cosine_similarity(tfidf_matrix)

In [48]:
import numpy as np

# Get the number of videos
n_videos = len(captions)

first_level_sim = (captions['first_level_category_name'].values[:, None] == captions['first_level_category_name'].values).astype(float)

second_level_sim = (captions['second_level_category_name'].values[:, None] == captions['second_level_category_name'].values).astype(float)

third_level_sim = (captions['third_level_category_name'].values[:, None] == captions['third_level_category_name'].values).astype(float)

# Weights for the similarities
text_weight = 0.5
first_level_weight = 0.3
second_level_weight = 0.15
third_level_weight = 0.05

# Combining the similarity matrices with weights
combined_sim = (
    text_sim * text_weight +
    first_level_weight * first_level_sim +
    second_level_weight * second_level_sim +
    third_level_weight * third_level_sim
)

combined_sim = np.clip(combined_sim, 0, 1)

In [49]:
indices = pd.Series(captions.index, index=captions['video_id']).drop_duplicates()

In [50]:
def get_enhanced_recommendations(video_id, indices, combined_sim, captions_df, num_recommend=10):
    """Get content-based recommendations for a video using the enhanced similarity matrix"""
    if video_id not in indices:
        return []
    
    idx = indices[video_id]
    
    if idx >= combined_sim.shape[0]:
        print(f"Index {idx} out of bounds for combined_sim with shape {combined_sim.shape}")
        return []
    
    sim_scores = list(enumerate(combined_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    top_similar = sim_scores[1:num_recommend+1]

    video_indices = [i[0] for i in top_similar]
    
    valid_indices = [i for i in video_indices if i < len(captions_df)]
    
    if len(valid_indices) < len(video_indices):
        print(f"Filtered out {len(video_indices) - len(valid_indices)} invalid indices")
    
    if not valid_indices:
        return []
        
    return captions_df['video_id'].iloc[valid_indices].tolist()

example_video_id = captions['video_id'].iloc[0]
enhanced_recommendations = get_enhanced_recommendations(example_video_id, indices, combined_sim, captions, num_recommend=5)
print(f"Enhanced recommendations for video {example_video_id}:")
print(enhanced_recommendations)

Enhanced recommendations for video 3649:
[5326, 8289, 6904, 1844, 6948]


In [51]:
def get_popular_recommendations(train_df, num_recommend=10):
    """Get most popular videos as fallback recommendations"""
    video_counts = train_df['video_id'].value_counts().reset_index()
    video_counts.columns = ['video_id', 'count']
    return video_counts.head(num_recommend)['video_id'].tolist()

In [52]:
captions = captions[['video_id', 'tokenized']]

train_df = pd.merge(train_df, captions, on='video_id', how='left')
test_df = pd.merge(test_df, captions, on='video_id', how='left')
train_user_hist = train_df.groupby("user_id")["video_id"].apply(list)
test_user_hist = test_df.groupby("user_id")["video_id"].apply(set)

fallback_recs = get_popular_recommendations(train_df, num_recommend=10)
top_k = 10
def precision_at_k(recommended_items, relevant_items, k):
    """Calculate precision@k"""
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    if not recommended_items:
        return 0.0
    
    hit = len(set(recommended_items) & set(relevant_items))
    return hit / min(k, len(recommended_items))

def recall_at_k(recommended_items, relevant_items, k):
    """Calculate recall@k"""
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    if not relevant_items:
        return 0.0
    
    hit = len(set(recommended_items) & set(relevant_items))
    return hit / len(relevant_items)

def ndcg_at_k(recommended_items, relevant_items, k):
    """Calculate nDCG@k"""
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    if not recommended_items or not relevant_items:
        return 0.0
    
    # Create a relevance list where 1 if the item is relevant, 0 otherwise
    relevance = [1 if item in relevant_items else 0 for item in recommended_items]
    
    # Calculate DCG
    dcg = 0
    for i, rel in enumerate(relevance):
        # i+1 because we're using 0-based indexing but rank is 1-based
        dcg += rel / np.log2(i + 2)  # log base 2 of rank+1
    
    # Calculate Ideal DCG (IDCG)
    ideal_relevance = [1] * min(len(relevant_items), k)
    idcg = 0
    for i, rel in enumerate(ideal_relevance):
        idcg += rel / np.log2(i + 2)
    
    return dcg / idcg if idcg > 0 else 0


def evaluate_enhanced_recommender(train_user_hist, test_user_hist, indices, combined_sim, captions_df, fallback_recs, top_k=10):
    """Evaluate recommender system with multiple metrics using the enhanced similarity"""
    hits = 0
    total = 0
    precision_sum = 0
    recall_sum = 0
    ndcg_sum = 0
    failed = 0
    fallback_used = 0
    
    # Keep track of missing video ids
    missing_video_ids = set()
    
    for user_id in test_user_hist.index:
        if user_id not in train_user_hist.index:
            continue
            
        test_videos = test_user_hist.get(user_id)
        train_videos = train_user_hist.get(user_id)
        if not train_videos or not test_videos:
            continue
        # Use the last video the user watched in train as seed
        seed_video = train_videos[-1]
        
        # Try content-based recommendations first
        recs = []
        used_fallback = False
        
        # Check if seed_video exists in indices
        if seed_video not in indices:
            missing_video_ids.add(seed_video)
            # Use fallback recommendations
            recs = fallback_recs
            used_fallback = True
        else:
            try:
                recs = get_enhanced_recommendations(seed_video, indices, combined_sim, captions_df, num_recommend=top_k)
                
                # If no recommendations, use fallback
                if not recs:
                    recs = fallback_recs
                    used_fallback = True
            except Exception as e:
                print(f"Error recommending for user {user_id}: {e}")
                recs = fallback_recs
                used_fallback = True
        
        if used_fallback:
            fallback_used += 1
            
        # Skip if still no recommendations
        if not recs:
            failed += 1
            continue
            
        # Calculate metrics
        if any(video in test_videos for video in recs):
            hits += 1
            
        precision = precision_at_k(recs, test_videos, top_k)
        recall = recall_at_k(recs, test_videos, top_k)
        ndcg = ndcg_at_k(recs, test_videos, top_k)
        
        precision_sum += precision
        recall_sum += recall
        ndcg_sum += ndcg
            
        total += 1
    
    print(f"Total users evaluated: {total}")
    print(f"Failed evaluations: {failed}")
    print(f"Fallback recommendations used: {fallback_used} ({fallback_used/total*100:.2f}% of total)")
    
    if missing_video_ids:
        print(f"Number of missing video IDs: {len(missing_video_ids)}")
        print(f"Sample missing video IDs: {list(missing_video_ids)[:5]}")
        
    # Return all metrics
    metrics = {
        f"HR@{top_k}": hits / total if total > 0 else 0,
        f"Precision@{top_k}": precision_sum / total if total > 0 else 0,
        f"Recall@{top_k}": recall_sum / total if total > 0 else 0,
        f"NDCG@{top_k}": ndcg_sum / total if total > 0 else 0,
        "Fallback_rate": fallback_used / total if total > 0 else 0
    }
    
    return metrics

enhanced_metrics = evaluate_enhanced_recommender(train_user_hist, test_user_hist, indices, combined_sim, captions, fallback_recs, top_k)
print("Enhanced Model Metrics:")
for metric, value in enhanced_metrics.items():
     print(f"{metric}: {value:.4f}")

Total users evaluated: 1411
Failed evaluations: 0
Fallback recommendations used: 0 (0.00% of total)
Enhanced Model Metrics:
HR@10: 0.9497
Precision@10: 0.6529
Recall@10: 0.0020
NDCG@10: 0.6602
Fallback_rate: 0.0000
